# R3 · English twins `_P` `_B` `_C` (Rathi)

**Run this only after the R2 notebook has finished and pushed.** This notebook needs R2's features and the reranker on Hugging Face Hub.

**Runtime → T4 GPU**, then **Run all** (about 1–1.5 h). It uses the same Colab Secrets as R2: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_KEY`.

Steps: 1 setup → 2 10-row checks → 3 full twins → 4 checks + samples → 5 twin features → 6 push.

In [ ]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push at the end.
from google.colab import userdata
import os
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R3-twins && git pull -q origin rathi/R3-twins
!git merge -q --no-edit origin/rathi/R2-reranker   # brings in features/ from the R2 run

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

!pip install -q -U wandb bitsandbytes

import subprocess, sys
SPACY = 'en_core_web_trf'
if subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY]).returncode != 0 or \
        subprocess.run([sys.executable, '-c', f'import spacy; spacy.load("{SPACY}")']).returncode != 0:
    SPACY = 'en_core_web_lg'
    subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY], check=True)
print('spaCy model:', SPACY)

import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)
from huggingface_hub import HfApi
RERANKER = f"{HfApi(token=os.environ['HF_TOKEN']).whoami()['name']}/twincheck-reranker-v1"
print('reranker:', RERANKER)
!ls -la features/

## 2. 10-row checks: 5 test + 5 train sources, all three twin types

In [ ]:
!python -m pytest tests/copies -q
!python -m src.copies.make_twins --n-test 5 --n-train 5 --reranker {RERANKER} --spacy-model {SPACY}     --out-dir data/copies/en/test10 --report data/copies/en/test10/report.json
!python -m src.copies.checks --twins data/copies/en/test10/test.jsonl data/copies/en/test10/train.jsonl     --log data/copies/en/test10/twin_log.csv --out data/copies/en/test10/checks.json     --samples-dir data/copies/en/test10/samples
!head -c 1500 data/copies/en/test10/test.jsonl

## 3. Full run: 500 test + 3,000 train sources (about 1 h, mostly Qwen paraphrasing)

In [ ]:
!python -m src.copies.make_twins --reranker {RERANKER} --spacy-model {SPACY}
print(open('results/R3/make_twins_report.json').read())

## 4. Automatic checks + 10 examples of each twin type

In [ ]:
!python -m src.copies.checks
import json
rep = json.load(open('results/R3/checks.json'))
for path, r in rep.items():
    print(path, r['per_type'], 'failed:', r['n_failed'], r['failures'][:3])

## 5. Relevance and entities for the twin snippets (Himasri's verdict head and R4 need these)

In [ ]:
!python -m src.retrieval.rerank data/copies/en/test.jsonl data/copies/en/train.jsonl     --model {RERANKER} --out features/relevance_copies_en.parquet
!python -m src.data.entities data/copies/en/test.jsonl data/copies/en/train.jsonl     --out features/entities_copies_en.jsonl --model {SPACY}
!du -h data/copies/en/* features/*copies*

## 6. Push to `rathi/R3-twins`

In [ ]:
!git add data/copies/en/test.jsonl data/copies/en/train.jsonl data/copies/en/sources.csv data/copies/en/twin_log.csv     features/relevance_copies_en.parquet features/entities_copies_en.jsonl     results/R3/make_twins_report.json results/R3/checks.json results/R3/samples
!git commit -q -m "[R3] Add English twins (_P _B _C), sources, checks and twin features from Colab run"
!git push -q origin rathi/R3-twins && echo PUSHED

**Done.** Tell Claude it finished, or paste the output of steps 3–4. Then **Runtime → Disconnect and delete runtime**.